In [1]:
import mujoco
import mujoco.viewer

import gtdynamics as gtd
import gtsam
import numpy
import plotly

In [ ]:
# Model paths
MUJOCO_MODEL = '../../models/mjcf/cart_pole.xml'
URDF_MODEL = '../../models/urdfs/cart_pole.urdf'

In [ ]:
# Load the MuJoCo model and launch interactive viewer
model = mujoco.MjModel.from_xml_path(MUJOCO_MODEL)
data = mujoco.MjData(model)

# Reset and initialize the simulation state
mujoco.mj_resetData(model, data)

mujoco.mj_forward(model, data)  # Compute forward dynamics

print("Model loaded successfully!")
# ... (rest of your script) ...

with mujoco.viewer.launch(model, data) as viewer:
    # Keep the viewer running until closed
    while viewer.is_running():
        # Step the simulation
        mujoco.mj_step(model, data)
        
        # Sync the viewer
        viewer.sync()

Model loaded successfully!


libdecor-gtk-WARNING: Failed to initialize GTK
Failed to load plugin 'libdecor-gtk.so': failed to init
No plugins found, falling back on no decorations
/home/havri/miniconda3/envs/gtd_sdf15/lib/python3.13/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


TypeError: 'NoneType' object does not support the context manager protocol

: 

In [7]:
# Create GTDynamics robot model from URDF
print("Loading robot model from URDF...")
robot = gtd.CreateRobotFromFile(URDF_MODEL)

# Print robot information
print(f"Number of links: {robot.numLinks()}")
print(f"Number of joints: {robot.numJoints()}")

# Print link information using link names
link_names = ["cart", "pole"]  # Based on URDF structure
for i, link_name in enumerate(link_names):
    try:
        link = robot.link(link_name)
        print(f"Link {i}: {link.name()}")
    except:
        print(f"Link {i}: {link_name} (not found)")

# Print joint information using joint names
joint_names = ["cart_joint", "pole_joint"]  # Based on URDF structure  
for i, joint_name in enumerate(joint_names):
    try:
        joint = robot.joint(joint_name)
        print(f"Joint {i}: {joint.name()}, type: {joint.type()}")
    except:
        print(f"Joint {i}: {joint_name} (not found)")

print("Robot model loaded successfully!")

Loading robot model from URDF...
Number of links: 2
Number of joints: 1
Link 0: cart
Link 1: pole
Joint 0: cart_joint (not found)
Joint 1: pole_joint, type: Type.Revolute
Robot model loaded successfully!


In [8]:
robot.joints()

[pole_joint (Revolute)
 	id=1
 	parent link: cart
 	child link: pole
 	screw axis (parent):   -0   -1   -0 0.05   -0   -0]

In [ ]:
# Set up trajectory optimization parameters
print("Setting up trajectory optimization...")

# Time parameters
total_time = 10.0  # seconds (5s swing-up + 5s stabilization)
dt = 0.05          # time step (20 Hz)
num_steps = int(total_time / dt)
swing_up_steps = int(5.0 / dt)  # First 5 seconds for swing-up

print(f"Total time: {total_time}s")
print(f"Time step: {dt}s") 
print(f"Number of steps: {num_steps}")
print(f"Swing-up phase: {swing_up_steps} steps")

# Fix the base link (cart moves on rail, not freely floating)
robot_fixed = robot.fixLink("l0")

# Gravity and dynamics setup - use default constructor for now
graph_builder = gtd.DynamicsGraph()

print("Robot and dynamics setup complete!")

In [ ]:
# Build the trajectory factor graph
print("Building factor graph...")

# Create trajectory factor graph using collocation
trajectory_graph = graph_builder.trajectoryFG(robot_fixed, num_steps, dt)

# Get joint indices
cart_joint_id = robot.joint("j0").id()  # prismatic cart joint
pole_joint_id = robot.joint("j1").id()  # revolute pole joint

print(f"Cart joint ID: {cart_joint_id}")
print(f"Pole joint ID: {pole_joint_id}")

# Initialize solution following the cart pole example pattern
print("Initializing trajectory values...")
initializer = gtd.Initializer()
init_values = initializer.ZeroValuesTrajectory(robot_fixed, num_steps, 0, 0.0, None)

print("Initial values set!")

In [ ]:
# Add boundary conditions and goals
print("Adding boundary conditions and goals...")

# Noise models for different constraints
dynamics_noise = gtsam.noiseModel.Isotropic.Sigma(1, 1e-6)
goal_noise = gtsam.noiseModel.Isotropic.Sigma(1, 1e-3)  # Relaxed goal noise
control_noise = gtsam.noiseModel.Isotropic.Sigma(1, 10.0)  # Relaxed control noise

# Initial state constraints (t=0): pole hanging down, cart at center
trajectory_graph.add(gtsam.PriorFactorDouble(gtd.JointAngleKey(pole_joint_id, 0), numpy.pi, dynamics_noise))
trajectory_graph.add(gtsam.PriorFactorDouble(gtd.JointAngleKey(cart_joint_id, 0), 0.0, dynamics_noise))
trajectory_graph.add(gtsam.PriorFactorDouble(gtd.JointVelKey(pole_joint_id, 0), 0.0, dynamics_noise))
trajectory_graph.add(gtsam.PriorFactorDouble(gtd.JointVelKey(cart_joint_id, 0), 0.0, dynamics_noise))

# Final goal state: pole upright (0), cart near center
final_t = num_steps
trajectory_graph.add(gtsam.PriorFactorDouble(gtd.JointAngleKey(pole_joint_id, final_t), 0.0, goal_noise))
trajectory_graph.add(gtsam.PriorFactorDouble(gtd.JointAngleKey(cart_joint_id, final_t), 0.0, goal_noise))
trajectory_graph.add(gtsam.PriorFactorDouble(gtd.JointVelKey(pole_joint_id, final_t), 0.0, goal_noise))
trajectory_graph.add(gtsam.PriorFactorDouble(gtd.JointVelKey(cart_joint_id, final_t), 0.0, goal_noise))

# Pole should be unactuated (no torque) - use constrained noise model
constrained_noise = gtsam.noiseModel.Constrained.All(1)
for t in range(num_steps + 1):
    trajectory_graph.add(gtsam.PriorFactorDouble(gtd.TorqueKey(pole_joint_id, t), 0.0, constrained_noise))

# Add control effort minimization for cart (minimum torque factors)
for t in range(num_steps):
    trajectory_graph.add(gtd.MinTorqueFactor(gtd.TorqueKey(cart_joint_id, t), control_noise))

print(f"Factor graph built with {trajectory_graph.size()} factors")
print("Boundary conditions and goals added!")

In [ ]:
# Optimize the factor graph
print("Starting optimization...")

# Set up Levenberg-Marquardt optimizer
params = gtsam.LevenbergMarquardtParams()
params.setMaxIterations(50)  # Reduced for faster computation
params.setAbsoluteErrorTol(1e-6)
params.setRelativeErrorTol(1e-6)
params.setVerbosity("ERROR")

optimizer = gtsam.LevenbergMarquardtOptimizer(trajectory_graph, init_values, params)

print("Running optimization...")
try:
    # Optimize
    optimized_values = optimizer.optimize()
    
    print("Optimization completed!")
    print(f"Initial error: {trajectory_graph.error(init_values)}")
    print(f"Final error: {trajectory_graph.error(optimized_values)}")
    print(f"Iterations: {optimizer.iterations()}")
    
    optimization_success = True
    
except Exception as e:
    print(f"Optimization failed: {e}")
    optimized_values = init_values
    optimization_success = False

In [ ]:
# Extract optimized trajectory
print("Extracting trajectory data...")

if optimization_success:
    values = optimized_values
else:
    values = init_values
    print("Using initial values due to optimization failure")

# Extract joint angles, velocities, and control inputs
times = [t * dt for t in range(num_steps + 1)]
cart_positions = []
pole_angles = []
cart_velocities = []
pole_velocities = []
cart_forces = []

for t in range(num_steps + 1):
    # Extract joint angles
    cart_pos = gtd.JointAngle(values, cart_joint_id, t)
    pole_angle = gtd.JointAngle(values, pole_joint_id, t)
    
    cart_positions.append(cart_pos)
    pole_angles.append(pole_angle)
    
    # Extract joint velocities
    cart_vel = gtd.JointVel(values, cart_joint_id, t)
    pole_vel = gtd.JointVel(values, pole_joint_id, t)
    
    cart_velocities.append(cart_vel)
    pole_velocities.append(pole_vel)
    
    # Extract control forces (torques) - only for num_steps, not num_steps+1
    if t < num_steps:
        cart_force = gtd.Torque(values, cart_joint_id, t)
        cart_forces.append(cart_force)

# Convert to numpy arrays
times = numpy.array(times)
cart_positions = numpy.array(cart_positions)
pole_angles = numpy.array(pole_angles)
cart_velocities = numpy.array(cart_velocities)
pole_velocities = numpy.array(pole_velocities)
cart_forces = numpy.array(cart_forces)

print(f"Trajectory extracted: {len(times)} time points")
print(f"Cart position range: [{cart_positions.min():.3f}, {cart_positions.max():.3f}]")
print(f"Pole angle range: [{pole_angles.min():.3f}, {pole_angles.max():.3f}]") 
print(f"Control force range: [{cart_forces.min():.3f}, {cart_forces.max():.3f}]")

In [ ]:
# Plot the optimized trajectory
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Joint angles over time
axes[0,0].plot(times, cart_positions, 'b-', linewidth=2, label='Cart Position')
axes[0,0].set_xlabel('Time (s)')
axes[0,0].set_ylabel('Cart Position (m)')
axes[0,0].set_title('Cart Position vs Time')
axes[0,0].grid(True)

axes[0,1].plot(times, pole_angles, 'r-', linewidth=2, label='Pole Angle')
axes[0,1].axhline(y=0, color='g', linestyle='--', alpha=0.5, label='Target (upright)')
axes[0,1].axhline(y=numpy.pi, color='orange', linestyle='--', alpha=0.5, label='Initial (hanging)')
axes[0,1].axvline(x=5, color='purple', linestyle='--', alpha=0.5, label='Swing-up end')
axes[0,1].set_xlabel('Time (s)')
axes[0,1].set_ylabel('Pole Angle (rad)')
axes[0,1].set_title('Pole Angle vs Time')
axes[0,1].legend()
axes[0,1].grid(True)

# Plot 2: Velocities
axes[1,0].plot(times, cart_velocities, 'b-', linewidth=2, label='Cart Velocity')
axes[1,0].set_xlabel('Time (s)')
axes[1,0].set_ylabel('Cart Velocity (m/s)')
axes[1,0].set_title('Cart Velocity vs Time')
axes[1,0].grid(True)

axes[1,1].plot(times, pole_velocities, 'r-', linewidth=2, label='Pole Velocity')
axes[1,1].set_xlabel('Time (s)')
axes[1,1].set_ylabel('Pole Angular Velocity (rad/s)')
axes[1,1].set_title('Pole Angular Velocity vs Time')
axes[1,1].grid(True)

plt.tight_layout()
plt.show()

# Plot control forces
plt.figure(figsize=(10, 4))
plt.plot(times[:-1], cart_forces, 'g-', linewidth=2)
plt.axvline(x=5, color='purple', linestyle='--', alpha=0.5, label='Swing-up end')
plt.xlabel('Time (s)')
plt.ylabel('Cart Force (N)')
plt.title('Control Force vs Time')
plt.legend()
plt.grid(True)
plt.show()

print("Trajectory plots complete!")

In [ ]:
# Create MuJoCo simulation with optimized trajectory
print("Setting up MuJoCo simulation...")

try:
    # Load MuJoCo model - now uses the matching cart-pole model
    mj_model = mujoco.MjModel.from_xml_path(MUJOCO_MODEL)
    mj_data = mujoco.MjData(mj_model)

    def apply_trajectory_to_mujoco(model, data, trajectory_times, cart_positions, pole_angles, control_forces):
        """Apply the optimized trajectory to MuJoCo simulation"""
        
        # Simulation parameters
        sim_time = 0.0
        framerate = 30  # fps
        frames = []
        
        # Interpolate trajectory for MuJoCo timestep
        mj_dt = model.opt.timestep
        trajectory_dt = trajectory_times[1] - trajectory_times[0]
        
        print(f"MuJoCo timestep: {mj_dt}")
        print(f"Trajectory timestep: {trajectory_dt}")
        print(f"MuJoCo joints: {[mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i) for i in range(model.njnt)]}")
        
        with mujoco.Renderer(model, width=600, height=400) as renderer:
            # Reset simulation
            mujoco.mj_resetData(model, data)
            
            while sim_time < total_time and len(frames) < 300:  # Limit frames
                # Find closest trajectory point
                traj_idx = min(int(sim_time / trajectory_dt), len(pole_angles) - 1)
                
                # Set joint positions to match optimized trajectory
                if traj_idx < len(pole_angles):
                    # Cart position (j0 - slide joint)
                    data.qpos[0] = cart_positions[traj_idx]
                    
                    # Pole angle (j1 - hinge joint)  
                    # GTDynamics: π = hanging, 0 = upright
                    # MuJoCo: 0 = upright, π = hanging (same convention)
                    data.qpos[1] = pole_angles[traj_idx]
                    
                    # Apply control force to cart
                    if traj_idx < len(control_forces):
                        data.ctrl[0] = control_forces[traj_idx]
                
                # Step simulation
                mujoco.mj_forward(model, data)  # Update derived quantities
                mujoco.mj_step(model, data)
                
                # Render frame occasionally
                if len(frames) < sim_time * framerate:
                    renderer.update_scene(data)
                    pixels = renderer.render()
                    frames.append(pixels)
                
                sim_time += mj_dt
        
        return frames

    # Create simulation frames - now with correct arguments
    print("Generating simulation frames...")
    sim_frames = apply_trajectory_to_mujoco(mj_model, mj_data, times, cart_positions, pole_angles, cart_forces)

    print(f"Generated {len(sim_frames)} simulation frames")

    # Display some key frames
    if len(sim_frames) > 0:
        import matplotlib.pyplot as plt
        
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        
        # Show frames at different time points
        time_points = [0, len(sim_frames)//4, len(sim_frames)//2, -1]
        labels = ['t=0s (hanging)', 't=2.5s', 't=5s', 't=10s (upright)']
        
        for i, (idx, label) in enumerate(zip(time_points, labels)):
            if idx < len(sim_frames) and i < len(axes):
                axes[i].imshow(sim_frames[idx])
                axes[i].set_title(label)
                axes[i].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        print("Key simulation frames displayed!")
    else:
        print("No frames generated - check simulation setup")

except Exception as e:
    print(f"MuJoCo simulation failed: {e}")
    print("This may be due to missing MuJoCo installation or model file issues.")
    print("Trajectory optimization was successful even without simulation visualization.")

In [ ]:
# Create video animation (optional - requires mediapy)
try:
    import mediapy as media
    
    if len(sim_frames) > 0:
        print("Creating video animation...")
        
        # Create video at 30 fps
        media.show_video(sim_frames, fps=30)
        print("Video animation created!")
    else:
        print("No frames available for video")
        
except ImportError:
    print("mediapy not available - install with: pip install mediapy")
    print("Showing alternative frame-by-frame animation...")
    
    if len(sim_frames) > 0:
        # Simple matplotlib animation
        from matplotlib.animation import FuncAnimation
        from IPython.display import HTML
        
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.set_title("Cart-Pole Swing-up Animation")
        ax.axis('off')
        
        im = ax.imshow(sim_frames[0])
        
        def animate(frame):
            if frame < len(sim_frames):
                im.set_array(sim_frames[frame])
            return [im]
        
        # Create animation (show every 3rd frame to speed up)
        anim = FuncAnimation(fig, animate, frames=range(0, len(sim_frames), 3), 
                           interval=100, blit=True, repeat=True)
        
        plt.show()
        print("Animation complete!")

print("\\nSummary:")
print("=" * 50)
print(f"✓ Robot model loaded from: {URDF_MODEL}")
print(f"✓ Factor graph built with {trajectory_graph.size()} factors")
print(f"✓ Optimization {'succeeded' if optimization_success else 'failed'}")
print(f"✓ Trajectory: {total_time}s, {num_steps} steps")
print(f"✓ Initial pole angle: {pole_angles[0]:.3f} rad (π = hanging)")
print(f"✓ Final pole angle: {pole_angles[-1]:.3f} rad (0 = upright)")
print(f"✓ Max cart force: {abs(cart_forces).max():.3f} N")
print(f"✓ Simulation: {len(sim_frames)} frames generated")

if optimization_success:
    print("\\n🎉 Cart-pole swing-up trajectory optimization completed successfully!")
else:
    print("\\n⚠️  Optimization failed - using initial trajectory")